# DRF Token Authentication

## Authorization Header

APIs authenticate requests using the `Authorization` HTTP header:

```
Authorization: <SCHEMA> <CREDENTIALS>
```

Common schemas:

| Schema | Used For |
|--------|----------|
| `Basic` | `base64(username:password)` — only safe with HTTPS |
| `Token` | DRF's built-in token auth |
| `Bearer` | JWT and OAuth2 |

Examples:
```bash
# Basic (demo only — always use HTTPS)
curl -H "Authorization: Basic dXNlcjpwYXNz" http://127.0.0.1:8000/api/ping/

# Token
curl -H "Authorization: Token abc123..." http://127.0.0.1:8000/api/accounts/
```


## BasicAuthentication (Demo)

```python
from rest_framework.authentication import BasicAuthentication
from rest_framework.permissions import IsAuthenticated
from rest_framework.views import APIView
from rest_framework.response import Response

class WhoAmIView(APIView):
    authentication_classes = [BasicAuthentication]
    permission_classes = [IsAuthenticated]

    def get(self, request):
        return Response({"user": request.user.username})
```

Do not use Basic authentication over plain HTTP — credentials are only base64-encoded, not encrypted.


## Enabling Token Authentication

```python
# settings.py
INSTALLED_APPS += ['rest_framework.authtoken']

REST_FRAMEWORK = {
    'DEFAULT_AUTHENTICATION_CLASSES': [
        'rest_framework.authentication.TokenAuthentication',
        'rest_framework.authentication.SessionAuthentication',
    ],
    'DEFAULT_PERMISSION_CLASSES': [
        'rest_framework.permissions.IsAuthenticated',
    ],
}
```

```bash
python manage.py migrate  # creates the authtoken_token table
```


## The Token Model

DRF's `Token` model has a **one-to-one relationship with User**. Each user has at most one token.

```python
from django.contrib.auth.models import User
from rest_framework.authtoken.models import Token

# Create a token for an existing user
user = User.objects.get(username='alice')
token, created = Token.objects.get_or_create(user=user)
print(token.key)  # the token string to use in the Authorization header
```

From the shell or Django admin, you can also use:
```bash
python manage.py drf_create_token <username>
```


## Built-in Token Login Endpoint

```python
# urls.py
from rest_framework.authtoken import views as authtoken_views

urlpatterns += [
    path('api-token-auth/', authtoken_views.obtain_auth_token, name='api-token-auth'),
]
```

```bash
POST /api-token-auth/
{"username": "alice", "password": "alicepw"}
# → {"token": "<key>"}
```

Then use the token in subsequent requests:
```
Authorization: Token <key>
```


## Custom Login and Logout Views

```python
from rest_framework.authtoken.views import ObtainAuthToken
from rest_framework.authtoken.models import Token
from rest_framework.views import APIView
from rest_framework.permissions import IsAuthenticated
from rest_framework.response import Response
from rest_framework import status

class CustomAuthToken(ObtainAuthToken):
    def post(self, request, *args, **kwargs):
        serializer = self.serializer_class(data=request.data, context={'request': request})
        serializer.is_valid(raise_exception=True)
        user = serializer.validated_data['user']
        token, _ = Token.objects.get_or_create(user=user)
        return Response({'token': token.key, 'user_id': user.pk, 'email': user.email})

class CustomAuthTokenLogout(APIView):
    permission_classes = [IsAuthenticated]

    def post(self, request, *args, **kwargs):
        request.user.auth_token.delete()   # revoke the token
        return Response(status=status.HTTP_204_NO_CONTENT)
```


## Testing Token Authentication

```python
from rest_framework.test import APITestCase
from rest_framework.authtoken.models import Token
from django.contrib.auth.models import User
from rest_framework import status

class TokenAuthTests(APITestCase):
    def setUp(self):
        self.alice = User.objects.create_user(username='alice', password='alicepw')
        self.alice_token = Token.objects.create(user=self.alice)

    def test_requires_auth(self):
        resp = self.client.get('/api/accounts/')
        self.assertEqual(resp.status_code, status.HTTP_401_UNAUTHORIZED)

    def test_list_with_token(self):
        self.client.credentials(HTTP_AUTHORIZATION=f'Token {self.alice_token.key}')
        resp = self.client.get('/api/accounts/')
        self.assertEqual(resp.status_code, status.HTTP_200_OK)

class ObtainTokenTests(APITestCase):
    def setUp(self):
        User.objects.create_user(username='carol', password='carolpw')

    def test_obtain_token(self):
        resp = self.client.post('/api/api-token-auth/', {'username': 'carol', 'password': 'carolpw'})
        self.assertEqual(resp.status_code, 200)
        self.assertIn('token', resp.data)
```


## Session Auth vs Token Auth

| Aspect | SessionAuthentication | TokenAuthentication |
|--------|----------------------|---------------------|
| Storage | Server-side session | Token string in DB |
| CSRF | Required for unsafe methods | Not required |
| Mobile / CORS | Difficult (cookie-based) | Simple (header-based) |
| Logout | Session destroy | Token delete |
| Best for | Browser-based same-origin apps | APIs, mobile clients |

## Summary

- Token authentication uses the `Authorization: Token <key>` header.
- The DRF `Token` model is one-to-one with `User`; each user has one token.
- Use the built-in `obtain_auth_token` view or a custom view to issue tokens on login.
- Revoke a token by deleting it from the database (`request.user.auth_token.delete()`).
- Token auth does not require CSRF, making it suitable for mobile and cross-origin clients.
